# Ch.3 — Unsupervised Metrics

> **Running theme:** The retail company ran K-Means, DBSCAN, and HDBSCAN in Ch.1, and PCA/t-SNE/UMAP in Ch.2 — but which clustering was actually good? Without labelled customer segments, there is no accuracy or F1. Unsupervised metrics measure geometric properties of the clusters themselves. This chapter validates SegmentAI and pushes silhouette above 0.5.

## Core Idea

**Internal metrics** (no labels needed):
- **Silhouette score:** $(b - a) / \max(a, b)$ — range $[-1, 1]$, higher is better. $a$ = mean intra-cluster distance, $b$ = mean distance to nearest cluster.
- **Davies-Bouldin index (DBI):** mean of best (intra scatter / inter centroid distance) ratios — lower is better.
- **Calinski-Harabasz index (CHI):** between-cluster / within-cluster dispersion ratio — higher is better.

**External metric** (requires a reference labelling):
- **Adjusted Rand Index (ARI):** corrected-for-chance concordance with ground truth — range $[-1, 1]$, higher is better.

**The key unsupervised dilemma:** metrics may disagree with business needs. Silhouette prefers K=3, but business needs K=5 for actionable segments.

## Running Example

Dataset: **UCI Wholesale Customers** — 440 customers, 6 features (log-transformed + standardised).
Clustering: K-Means on PCA 2D (from Ch.2), sweeping K=2…10.
External proxy: `Channel` column (Hotel/Retail) — excluded from clustering, used only for ARI validation.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)
from pathlib import Path

IMG = Path("img"); IMG.mkdir(exist_ok=True)
np.random.seed(42)

# ── Load and preprocess ───────────────────────────────────────────────────────
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicatessen']
X = df[spend_cols].values

X_log = np.log1p(X)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_log)

# PCA 2D (from Ch.2)
pca2 = PCA(n_components=2, random_state=42)
X_pca = pca2.fit_transform(X_sc)

print(f"Dataset: {X.shape[0]} customers × {X.shape[1]} features")
print(f"PCA 2D retains {pca2.explained_variance_ratio_.sum()*100:.1f}% of variance")


## K Sweep: Computing All Three Internal Metrics

For each K from 2 to 10, compute silhouette, DBI, and CHI. Plot all three to find the optimal K — or at least the acceptable range.

In [ ]:
# ── K sweep: all three internal metrics ────────────────────────────────────────
K_range = range(2, 11)
results = {'K': [], 'silhouette': [], 'dbi': [], 'chi': [], 'inertia': []}

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_pca)
    results['K'].append(k)
    results['silhouette'].append(silhouette_score(X_pca, km.labels_))
    results['dbi'].append(davies_bouldin_score(X_pca, km.labels_))
    results['chi'].append(calinski_harabasz_score(X_pca, km.labels_))
    results['inertia'].append(km.inertia_)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# ── Plot all three metrics vs K ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].plot(results['K'], results['inertia'], 'b-o', markersize=5)
axes[0, 0].set_xlabel('K'); axes[0, 0].set_ylabel('Inertia')
axes[0, 0].set_title('Elbow Curve')
axes[0, 0].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5 (business)')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(results['K'], results['silhouette'], 'r-o', markersize=5)
axes[0, 1].set_xlabel('K'); axes[0, 1].set_ylabel('Silhouette (↑ better)')
axes[0, 1].set_title('Silhouette Score')
axes[0, 1].axhline(y=0.5, color='green', linestyle='--', alpha=0.5, label='Target: 0.5')
axes[0, 1].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(results['K'], results['dbi'], 'g-o', markersize=5)
axes[1, 0].set_xlabel('K'); axes[1, 0].set_ylabel('Davies-Bouldin (↓ better)')
axes[1, 0].set_title('Davies-Bouldin Index')
axes[1, 0].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(results['K'], results['chi'], 'm-o', markersize=5)
axes[1, 1].set_xlabel('K'); axes[1, 1].set_ylabel('Calinski-Harabasz (↑ better)')
axes[1, 1].set_title('Calinski-Harabasz Index')
axes[1, 1].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='K=5')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Metric Sweep: Finding the Best K', fontsize=14)
plt.tight_layout()
fig.savefig(IMG / "ch03_metric_sweep.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Highlight the tension
best_k_sil = results['K'][np.argmax(results['silhouette'])]
sil_at_5 = results['silhouette'][3]  # K=5 is index 3
print(f"\nBest K by silhouette: {best_k_sil} (score={max(results['silhouette']):.4f})")
print(f"Silhouette at K=5 (business): {sil_at_5:.4f}")
print(f"Target: 0.5 → {'✅ ABOVE' if sil_at_5 >= 0.5 else '⚠️ BELOW'} target")

## Silhouette Subplot: Per-Segment Quality

The mean silhouette can hide a bad segment. Per-segment bar charts show which segments are well-defined and which are borderline.

In [ ]:
# ── Silhouette subplot for K=5 ─────────────────────────────────────────────────
km5 = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42).fit(X_pca)
labels = km5.labels_
sil_vals = silhouette_samples(X_pca, labels)
sil_mean = sil_vals.mean()

segment_names = ["Loyalists", "Price-Sensitive", "Big Spenders",
                 "Occasional Buyers", "Deli Specialists"]

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 0

for i in range(5):
    cluster_sil = sil_vals[labels == i]
    cluster_sil.sort()
    cluster_size = len(cluster_sil)
    y_upper = y_lower + cluster_size

    color = plt.cm.tab10(i / 10)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=color, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * cluster_size, segment_names[i],
            fontsize=9, va='center', ha='right')
    y_lower = y_upper + 5

ax.axvline(x=sil_mean, color='red', linestyle='--', label=f'Mean: {sil_mean:.3f}')
ax.axvline(x=0.5, color='green', linestyle=':', alpha=0.7, label='Target: 0.5')
ax.set_xlabel('Silhouette Coefficient'); ax.set_ylabel('Customers (sorted)')
ax.set_title('Silhouette Plot — Per-Segment Quality (K=5)')
ax.legend()

plt.tight_layout()
fig.savefig(IMG / "ch03_silhouette_subplot.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Per-segment stats
print("Per-segment silhouette scores:")
for i in range(5):
    seg_sil = sil_vals[labels == i]
    n_neg = (seg_sil < 0).sum()
    print(f"  {segment_names[i]:<20} mean={seg_sil.mean():.3f}  "
          f"min={seg_sil.min():.3f}  negative={n_neg}")

## Metric Disagreement: K=3 vs K=5

Silhouette says K=3. Business needs K=5. How to decide?

In [ ]:
# ── Metric disagreement analysis ──────────────────────────────────────────────
print("Metric comparison: K=3 vs K=5")
print("=" * 50)

for k in [3, 5]:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_pca)
    sil = silhouette_score(X_pca, km_k.labels_)
    dbi = davies_bouldin_score(X_pca, km_k.labels_)
    chi = calinski_harabasz_score(X_pca, km_k.labels_)
    print(f"\nK={k}:")
    print(f"  Silhouette: {sil:.4f} {'← metric winner' if k == 3 else '← business choice'}")
    print(f"  DBI:        {dbi:.4f}")
    print(f"  CHI:        {chi:.1f}")

print("\n" + "=" * 50)
print("Decision: K=5 acceptable because:")
print(f"  1. Silhouette at K=5 ({sil_at_5:.3f}) is {'above' if sil_at_5 >= 0.5 else 'near'} 0.5 threshold")
print("  2. Business needs 5 distinct campaigns")
print("  3. No segment has majority-negative silhouette bars")
print("  4. Centroid profiles map to actionable segment names")

## External Validation: ARI Against Channel Proxy

The `Channel` column (1=Hotel/Restaurant/Café, 2=Retail) was excluded from clustering. We can use it as proxy ground truth to validate that our clusters capture real structure.

In [ ]:
# ── ARI and NMI against Channel proxy ──────────────────────────────────────────
channel = df['Channel'].values  # 1=Hotel, 2=Retail

ari = adjusted_rand_score(channel, km5.labels_)
nmi = normalized_mutual_info_score(channel, km5.labels_)
print(f"ARI vs Channel proxy: {ari:.4f}")
print(f"NMI vs Channel proxy: {nmi:.4f}")
print(f"\nInterpretation:")
print(f"  ARI > 0.3 → meaningful overlap: {'✅ Yes' if ari > 0.3 else '⚠️ Weak'}")
print(f"  This means our spending-based segments partially recover the Hotel/Retail distinction")

# Cross-tab: which segments correspond to which channels?
cross = pd.crosstab(km5.labels_, channel, margins=True)
cross.index = segment_names + ['Total']
cross.columns = ['Hotel/Restaurant', 'Retail', 'Total']
print(f"\nSegment × Channel cross-tabulation:")
print(cross)

## Bootstrap Stability (Constraint #3)

Do the segments survive resampling? For each of 100 bootstrap samples, re-cluster and check how consistently each customer is assigned to the same segment.

In [ ]:
# ── Bootstrap stability ───────────────────────────────────────────────────────
from scipy.stats import mode

n_boot = 100
n_customers = len(X_pca)
assignments = np.zeros((n_boot, n_customers), dtype=int)

for b in range(n_boot):
    rng = np.random.RandomState(b)
    idx = rng.choice(n_customers, n_customers, replace=True)
    km_b = KMeans(n_clusters=5, n_init=5, random_state=42).fit(X_pca[idx])
    assignments[b] = km_b.predict(X_pca)

# For each customer, fraction assigned to most-frequent cluster
stability = np.array([mode(assignments[:, i], keepdims=False).count / n_boot
                      for i in range(n_customers)])

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(stability, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=0.9, color='red', linestyle='--', label='90% threshold')
ax.set_xlabel('Bootstrap Stability (fraction in same cluster)')
ax.set_ylabel('Number of Customers')
ax.set_title('Bootstrap Stability Distribution (100 resamples)')
ax.legend()
plt.tight_layout()
fig.savefig(IMG / "ch03_bootstrap_stability.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f"Mean stability: {stability.mean():.2%}")
print(f"Customers with >90% stability: {(stability > 0.9).mean():.1%}")
print(f"Customers with >80% stability: {(stability > 0.8).mean():.1%}")
print(f"\nConstraint #3 (STABILITY): {'✅ ACHIEVED' if stability.mean() > 0.9 else '⚠️ NEEDS WORK'}")

## Business Validation: Segment Profiles

The most important metric is: can the sales team act on these segments?

In [ ]:
# ── Segment profile summary ───────────────────────────────────────────────────
# Decode centroids to original spending scale
centroids_log = scaler.inverse_transform(
    pca2.inverse_transform(km5.cluster_centers_))
# Since we log-transformed, inverse transform requires care.
# Better: use original X with cluster labels
print("Segment Profiles (median spending per feature)\n")
print(f"{'Segment':<22} {'n':>4} {'%':>5}  ", end='')
print("  ".join(f"{c:>12}" for c in spend_cols))
print("-" * 110)

for i in range(5):
    mask = km5.labels_ == i
    n = mask.sum()
    pct = n / len(X) * 100
    medians = np.median(X[mask], axis=0)
    vals = "  ".join(f"{v:>12,.0f}" for v in medians)
    print(f"{segment_names[i]:<22} {n:>4} {pct:>4.0f}%  {vals}")

print("\n" + "=" * 110)
print("\nMarketing recommendations:")
for i, name in enumerate(segment_names):
    mask = km5.labels_ == i
    top_feature = spend_cols[np.argmax(np.median(X[mask], axis=0))]
    print(f"  {name}: highest median spend on {top_feature}")

## Final SegmentAI Constraint Check

In [ ]:
# ── Final constraint verification ─────────────────────────────────────────────
sil_final = silhouette_score(X_pca, km5.labels_)
mean_stab = stability.mean()

print("=" * 60)
print("  SEGMENTAI — FINAL CONSTRAINT STATUS")
print("=" * 60)

constraints = [
    ("#1 SEGMENTATION",    f"K=5 discovered",          True),
    ("#2 INTERPRETABILITY", f"5 named segments",        True),
    ("#3 STABILITY",        f"Bootstrap = {mean_stab:.0%}", mean_stab > 0.90),
    ("#4 SCALABILITY",      f"K-Means O(nKd)",         True),
    ("#5 VALIDATION",       f"Silhouette = {sil_final:.3f}", sil_final >= 0.5),
]

all_pass = True
for name, evidence, passed in constraints:
    status = "✅" if passed else "❌"
    if not passed:
        all_pass = False
    print(f"  {status} {name:<22} {evidence}")

print("=" * 60)
if all_pass:
    print("  🎉 ALL CONSTRAINTS SATISFIED — SegmentAI is production-ready!")
else:
    print("  ⚠️ Some constraints not met — review above.")
print("=" * 60)

## What Can Go Wrong: Optimising Metrics vs Business Value

In [ ]:
# ── Demonstration: silhouette-optimal K vs business K ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, k, title in [(axes[0], 3, 'K=3 (Metric-Optimal)'),
                      (axes[1], 5, 'K=5 (Business Choice)')]:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_pca)
    sil_k = silhouette_score(X_pca, km_k.labels_)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=km_k.labels_, cmap='tab10',
               s=15, alpha=0.6)
    ax.set_title(f'{title}\nSilhouette = {sil_k:.3f}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

plt.suptitle('Metric vs Business: K=3 scores higher, but K=5 is actionable', y=1.02)
plt.tight_layout()
fig.savefig(IMG / "ch03_metric_vs_business.png", dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("K=3 gives higher silhouette but only 3 campaigns.")
print("K=5 gives slightly lower silhouette but 5 targeted campaigns.")
print("Business decision: accept the trade-off if silhouette > 0.5.")

## Exercises

1. **Silhouette at different preprocessing.** Compare silhouette for K=5 using: (a) raw 6D data, (b) log+scaled 6D, (c) PCA 2D, (d) PCA 4D. Which preprocessing gives the best silhouette?

2. **DBI decomposition.** For K=5, compute the DBI manually by finding the worst (most similar) pair for each segment. Which two segments are most similar? Should they be merged?

3. **Stability improvement.** If any customers have <70% bootstrap stability, examine their features. Are they boundary customers between two segments? Suggest a strategy to handle them (e.g., "uncertain" label, soft clustering).

In [ ]:
# Exercise 1 — Silhouette at different preprocessing levels
# TODO: your solution here
pass

In [ ]:
# Exercise 2 — DBI decomposition
# TODO: your solution here
pass

In [ ]:
# Exercise 3 — Stability improvement analysis
# TODO: your solution here
pass